# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [1]:
# 1-2, 4-5, 

class matrix:

    def __init__(self, n, m=None):
        if isinstance(n, list) and m is None:
            self.data = self._copy_rows(n)
            self.n = len(self.data)
            self.m = len(self.data[0]) if self.n else 0

        elif type(n) is int and type(m) is int:
            if n < 0 or m < 0:
                raise ValueError("Matrix dimensions must be nonnegative.")

            self.n, self.m = n, m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        else:
            raise TypeError("Use matrix(n, m) or matrix(list_of_lists).")

    @staticmethod
    def _copy_rows(values):
        if not isinstance(values, list) or not all(
            isinstance(row, list) for row in values
        ):
            raise TypeError("Matrix values must be a list of lists.")

        width = len(values[0]) if values else 0

        if any(len(row) != width for row in values):
            raise ValueError("All rows must have the same number of columns.")

        return [row[:] for row in values]

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        result = matrix(self.m, self.n)

        for i in range(self.n):
            for j in range(self.m):
                result.data[j][i] = self.data[i][j]

        return result

    def row(self, n):
        return self[n, :]

    def column(self, n):
        return self[:, n]

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return self[m_0:m_1, n_0:n_1]

    def __getitem__(self, key):
        if not isinstance(key, (tuple, slice)):
            return self.data[key]

        if isinstance(key, tuple):
            row_key, col_key = key

            if (
                not isinstance(row_key, slice)
                and not isinstance(col_key, slice)
            ):
                return self.data[row_key][col_key]

        else:
            row_key = key
            col_key = slice(None)

        def indices(index, size):
            if isinstance(index, slice):
                return list(range(size))[index]

            if not isinstance(index, int):
                raise TypeError("Indices must be integers or slices.")

            if index < 0:
                index += size

            if index < 0 or index >= size:
                raise IndexError("Matrix index out of range.")

            return [index]

        row_indices = indices(row_key, self.n)
        col_indices = indices(col_key, self.m)

        result = matrix(len(row_indices), len(col_indices))

        for i, source_i in enumerate(row_indices):
            for j, source_j in enumerate(col_indices):
                result.data[i][j] = self.data[source_i][source_j]

        return result

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value

        else:
            if not isinstance(value, list) or len(value) != self.m:
                raise ValueError("Replacement row must have the correct length.")

            self.data[key] = value[:]

    def assign(self, other):
        if isinstance(other, matrix):
            rows = self._copy_rows(other.data)
            size = (other.n, other.m)

        else:
            rows = self._copy_rows(other)
            size = (len(rows), len(rows[0]) if rows else 0)

        if size != (self.n, self.m):
            raise ValueError("Assignment requires matrices of the same size.")

        self.data = rows


    def _check_same_shape(self, other):
        if not isinstance(other, matrix):
            raise TypeError("The argument must be a matrix.")

        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same dimensions.")

    def scalarmul(self, c):
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result.data[i][j] = c * self.data[i][j]

        return result

    def add(self, N):
        self._check_same_shape(N)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result.data[i][j] = self.data[i][j] + N.data[i][j]

        return result

    def sub(self, N):
        self._check_same_shape(N)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result.data[i][j] = self.data[i][j] - N.data[i][j]

        return result

    def mat_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("The argument must be a matrix.")

        if self.m != N.n:
            raise ValueError(
                "The first matrix must have as many columns "
                "as the second has rows."
            )

        result = matrix(self.n, N.m)

        for i in range(self.n):
            for j in range(N.m):
                for k in range(self.m):
                    result.data[i][j] += (
                        self.data[i][k] * N.data[k][j]
                    )

        return result

    def element_mult(self, N):
        self._check_same_shape(N)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result.data[i][j] = self.data[i][j] * N.data[i][j]

        return result

    def equals(self, N):
        if not isinstance(N, matrix):
            return False

        return self.shape() == N.shape() and self.data == N.data

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)

        if isinstance(other, (int, float, complex)):
            return self.scalarmul(other)

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (int, float, complex)):
            return self.scalarmul(other)

        return NotImplemented

    def __add__(self, other):
        if isinstance(other, matrix):
            return self.add(other)

        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, matrix):
            return self.sub(other)

        return NotImplemented

    def __eq__(self, other):
        return self.equals(other)

In [2]:
# 1

Z = matrix(2, 3)
assert Z.data == [[0, 0, 0], [0, 0, 0]]

Z[0][0] = 9
assert Z[1][0] == 0


values = [[1, 2], [3, 4]]
M = matrix(values)
assert M.data == [[1, 2], [3, 4]]

values[0][0] = 99
assert M[0][0] == 1


assert M[1][0] == 3
assert M[1, 0] == 3

M[0][1] = 8
M[1, 1] = 9
assert M.data == [[1, 8], [3, 9]]

M[0] = [5, 6]
assert M[0] == [5, 6]


M1 = matrix(2, 2)
M2 = matrix([[10, 20], [30, 40]])

M1.assign(M2)
assert M1.data == [[10, 20], [30, 40]]

M2[0, 0] = 100
assert M1[0, 0] == 10


new_values = [[5, 6], [7, 8]]
M1.assign(new_values)
assert M1.data == [[5, 6], [7, 8]]

new_values[0][0] = 100
assert M1[0, 0] == 5


def expect_error(error_type, action):
    try:
        action()
    except error_type:
        pass
    else:
        raise AssertionError(f"Expected {error_type.__name__}")


expect_error(ValueError, lambda: matrix([[1, 2], [3]]))
expect_error(TypeError, lambda: matrix([1, 2, 3]))
expect_error(ValueError, lambda: matrix(-1, 2))
expect_error(TypeError, lambda: matrix(2.5, 2))
expect_error(ValueError, lambda: M1.assign(matrix(1, 2)))
expect_error(ValueError, lambda: M1.assign([[1, 2, 3], [4, 5, 6]]))

expect_error(ValueError, lambda: M1.assign([[1, 2], [3]]))

expect_error(ValueError, lambda: M1.__setitem__(0, [1]))

assert M1.data == [[5, 6], [7, 8]]

assert matrix(0, 3).data == []
assert matrix([]).data == []
assert matrix(2, 0).data == [[], []]

print("passed!")

passed!


In [3]:
# 2
M = matrix([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
])

assert M.shape() == (3, 4)


T = M.transpose()

assert isinstance(T, matrix)
assert T.shape() == (4, 3)
assert T.to_list() == [
    [1, 5, 9],
    [2, 6, 10],
    [3, 7, 11],
    [4, 8, 12]
]

assert T.transpose().to_list() == M.to_list()


R = M.row(1)

assert isinstance(R, matrix)
assert R.shape() == (1, 4)
assert R.to_list() == [[5, 6, 7, 8]]

C = M.column(2)

assert isinstance(C, matrix)
assert C.shape() == (3, 1)
assert C.to_list() == [[3], [7], [11]]


values = M.to_list()

assert values == [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
]

values[0][0] = 99
assert M[0, 0] == 1


B = M.block(1, 3, 0, 2)

assert B.shape() == (2, 2)
assert B.to_list() == [[2, 3], [6, 7]]



assert M[1][2] == 7
assert M[1, 2] == 7
assert M[0:2].to_list() == [
    [1, 2, 3, 4],
    [5, 6, 7, 8]
]

assert M[0:2, 1:3].to_list() == [[2, 3], [6, 7]]
assert M[1, :].to_list() == [[5, 6, 7, 8]]
assert M[:, 2].to_list() == [[3], [7], [11]]
assert M[::2, ::2].to_list() == [[1, 3], [9, 11]]
assert M[-1, :].to_list() == [[9, 10, 11, 12]]
assert M[0:0, :].shape() == (0, 4)
assert M[:, 0:0].shape() == (3, 0)


S = M[:, :]

for result in (T, R, C, B, S):
    result[0][0] = -100

assert M.to_list() == [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
]

print("All Question 2 tests passed!")

All Question 2 tests passed!


In [4]:
# 3

def constant(n, m, c):
    result = matrix(n, m)
    value = float(c)

    for i in range(n):
        for j in range(m):
            result[i, j] = value

    return result


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    result = zeros(n, n)

    for i in range(n):
        result[i, i] = 1.0

    return result



C = constant(2, 3, 7)
assert C.shape() == (2, 3)
assert C.to_list() == [
    [7.0, 7.0, 7.0],
    [7.0, 7.0, 7.0]
]

assert constant(1, 2, -2.5).to_list() == [[-2.5, -2.5]]

Z = zeros(2, 3)
assert Z.shape() == (2, 3)
assert Z.to_list() == [
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0]
]

O = ones(3, 2)
assert O.shape() == (3, 2)
assert O.to_list() == [
    [1.0, 1.0],
    [1.0, 1.0],
    [1.0, 1.0]
]

I = eye(3)
assert I.shape() == (3, 3)
assert I.to_list() == [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
]

for result in (C, Z, O, I):
    assert isinstance(result, matrix)
    assert all(
        isinstance(value, float)
        for row in result.to_list()
        for value in row
    )

C[0, 0] = 99.0
assert C[1, 0] == 7.0

print("Question 3 tests passed!")

Question 3 tests passed!


In [5]:
# 4

A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])


assert A.scalarmul(2).to_list() == [[2, 4], [6, 8]]

assert A.scalarmul(-0.5).to_list() == [
    [-0.5, -1.0],
    [-1.5, -2.0]
]


# (b) Test addition.
assert A.add(B).to_list() == [[6, 8], [10, 12]]


assert A.sub(B).to_list() == [[-4, -4], [-4, -4]]


assert A.mat_mult(B).to_list() == [[19, 22], [43, 50]]

P = matrix([
    [1, 2, 3],
    [4, 5, 6]
])

Q = matrix([
    [7, 8],
    [9, 10],
    [11, 12]
])

PQ = P.mat_mult(Q)

assert PQ.shape() == (2, 2)
assert PQ.to_list() == [[58, 64], [139, 154]]


assert A.element_mult(B).to_list() == [[5, 12], [21, 32]]


assert A.equals(matrix([[1, 2], [3, 4]])) is True
assert A.equals(B) is False
assert A.equals(matrix(1, 4)) is False
assert A.equals([[1, 2], [3, 4]]) is False


def expect_error(error_type, action):
    try:
        action()
    except error_type:
        pass
    else:
        raise AssertionError(f"Expected {error_type.__name__}")


for operation in (A.add, A.sub, A.element_mult):
    expect_error(
        ValueError,
        lambda: operation(matrix(2, 3))
    )

expect_error(
    ValueError,
    lambda: A.mat_mult(matrix(3, 2))
)

for operation in (A.add, A.sub, A.mat_mult, A.element_mult):
    expect_error(
        TypeError,
        lambda: operation([[1, 2], [3, 4]])
    )


results = [
    A.scalarmul(2),
    A.add(B),
    A.sub(B),
    A.mat_mult(B),
    A.element_mult(B)
]

for result in results:
    assert isinstance(result, matrix)
    result[0, 0] = 999

assert A.to_list() == [[1, 2], [3, 4]]
assert B.to_list() == [[5, 6], [7, 8]]

print("All Question 4 tests passed!")

All Question 4 tests passed!


In [6]:
# 5

M = matrix([[1, 2], [3, 4]])
N = matrix([[5, 6], [7, 8]])


assert (2 * M).to_list() == [[2, 4], [6, 8]]


assert (M * 2).to_list() == [[2, 4], [6, 8]]

assert (2 * M) == (M * 2)


assert (M + N).to_list() == [[6, 8], [10, 12]]


assert (M - N).to_list() == [[-4, -4], [-4, -4]]


assert (M * N).to_list() == [[19, 22], [43, 50]]


assert (M == matrix([[1, 2], [3, 4]])) is True
assert (M == N) is False
assert (M == matrix(1, 4)) is False
assert (M == [[1, 2], [3, 4]]) is False


def expect_error(error_type, action):
    try:
        action()
    except error_type:
        pass
    else:
        raise AssertionError(f"Expected {error_type.__name__}")


expect_error(ValueError, lambda: M + matrix(2, 3))
expect_error(ValueError, lambda: M - matrix(2, 3))
expect_error(ValueError, lambda: M * matrix(3, 2))

expect_error(TypeError, lambda: M + 2)
expect_error(TypeError, lambda: M * "hello")
expect_error(TypeError, lambda: "hello" * M)

assert M.to_list() == [[1, 2], [3, 4]]
assert N.to_list() == [[5, 6], [7, 8]]

print("Question 5 passed")

Question 5 passed


In [7]:
# 6

A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [1, 0]])
C = matrix([[2, 0], [0, 3]])

I = eye(2)


left = (A * B) * C
right = A * (B * C)

print("(AB)C =", left.to_list())
print("A(BC) =", right.to_list())

assert left == right
assert left.to_list() == [[4, 3], [8, 9]]


left = A * (B + C)
right = (A * B) + (A * C)

print("\nA(B + C) =", left.to_list())
print("AB + AC  =", right.to_list())

assert left == right
assert left.to_list() == [[4, 7], [10, 15]]


AB = A * B
BA = B * A

print("\nAB =", AB.to_list())
print("BA =", BA.to_list())

assert AB.to_list() == [[2, 1], [4, 3]]
assert BA.to_list() == [[3, 4], [1, 2]]
assert not (AB == BA)


AI = A * I

print("\nAI =", AI.to_list())
print("A  =", A.to_list())

assert AI == A


print("passed question 6")

(AB)C = [[4, 3], [8, 9]]
A(BC) = [[4, 3], [8, 9]]

A(B + C) = [[4, 7], [10, 15]]
AB + AC  = [[4, 7], [10, 15]]

AB = [[2, 1], [4, 3]]
BA = [[3, 4], [1, 2]]

AI = [[1.0, 2.0], [3.0, 4.0]]
A  = [[1, 2], [3, 4]]
passed question 6
